In [1]:
!pip install -q torch transformers peft trl bitsandbytes datasets wandb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 9.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch

In [3]:
import wandb
import os
import shutil
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN") 
login(token=hf_token)

In [5]:
wandb.login()
wandb.init(project="industrial-defect-summarizer", name="kaggle-t4-run")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krustykunal (krustykunal-manipal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
dataset = load_dataset("json", data_files="/kaggle/input/datasets/kunalgzol/defect-fine-tuning/defects_train.jsonl", split="train")
model_id = "NousResearch/Meta-Llama-3-8B-Instruct"

In [14]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [16]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [17]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [18]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [22]:
training_args = SFTConfig(
    output_dir="/kaggle/working/llama-3-8b-defect-lora",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    save_steps=30,
    logging_steps=10,
    learning_rate=2e-4,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=150,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    push_to_hub=True,
    hub_model_id="KunalG-Zol/llama-3-8b-defect-summarizer",
    hub_private_repo=True,
    report_to="wandb",
    save_total_limit=3
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [23]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Truncating train dataset:   0%|          | 0/52563 [00:00<?, ? examples/s]

In [24]:
print("Starting training loop...")
trainer.train()

print("Training complete. Saving and zipping weights...")

Starting training loop...


Step,Training Loss
10,3.125021
20,2.416412
30,2.272226
40,2.217769
50,2.173604
60,2.215156
70,2.092743
80,2.172603
90,2.170496
100,2.088073


Training complete. Saving and zipping weights...


In [25]:
trainer.model.save_pretrained("./final_adapter")
wandb.finish()
shutil.make_archive('defect_lora_weights', 'zip', '/kaggle/working/final_adapter')
print("Training finished. Weights pushed to Hugging Face and zipped locally.")

train/entropy,██▅▃▃▄▁▃▂▁▁▃▂▃▂
train/epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇███
train/global_step,▁▁▂▃▃▃▄▅▅▅▆▇▇▇██
train/grad_norm,█▃▂▃▃▁▃▃▁▃▃▃▂▂▃
train/learning_rate,███▇▇▆▅▄▄▃▂▂▁▁▁
train/loss,█▃▂▂▂▂▁▂▂▁▁▁▁▂▁
train/mean_token_accuracy,▁▅▆▇▇▇█▇▇███▇▇█
train/num_tokens,▁▁▂▂▃▃▄▄▅▅▆▆▇██
total_flos,1.861794498809856e+16
train/entropy,2.12055
train/epoch,0.02283


Training finished. Weights pushed to Hugging Face and zipped locally.


In [26]:
import shutil
shutil.make_archive('best_lora_weights', 'zip', '/kaggle/working/llama-3-8b-defect-lora/checkpoint-100')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/llama-3-8b-defect-lora/checkpoint-100'